<a href="https://colab.research.google.com/github/R-SamiUllah/Flyrank-Task1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/R-SamiUllah/Flyrank-Task1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN[:10])

hf_vbxMsbC


In [2]:
!pip -q install duckdb huggingface_hub

In [3]:
from huggingface_hub import login

login(token=HF_TOKEN)

print("Logged in successfully")

Logged in successfully


In [4]:
from datasets import load_dataset

content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content"
)

In [5]:
content_df = content["train"].to_pandas()

In [6]:
content_df.head()

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


In [7]:
performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [8]:
march_2026 = performance["train"].select(
    range(35800000, 45000000)
)

In [9]:
performance_df = march_2026.to_pandas()

In [10]:
performance_df.shape

(9200000, 30)

In [11]:
performance_df["report_date"].min(), performance_df["report_date"].max()

(datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))

In [12]:
performance["train"].select(range(45000000,45000008))["report_date"]

Column([datetime.date(2026, 3, 31), datetime.date(2026, 3, 31), datetime.date(2026, 3, 31), datetime.date(2026, 3, 31), datetime.date(2026, 3, 31)])

In [13]:
performance["train"].select(range(35800000,35800008))["report_date"]


Column([datetime.date(2026, 3, 1), datetime.date(2026, 3, 1), datetime.date(2026, 3, 1), datetime.date(2026, 3, 1), datetime.date(2026, 3, 1)])

In [14]:
performance_df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

##1. Unit of analysis

One row represents the daily search and analytics performance of one content item for one client on one report date.

The grain is:

(client_hash_id, content_hash_id, report_date)
This means each record captures how one content item performed for one client on a specific day.
## Time window

For this assignment I use the March 2026 partition.

The selected period is:
2026-03-01 to 2026-03-31

This is a historical panel month used to build and verify features before making refresh decisions.

In [15]:
# Check the grain of the dataset

grain_check = performance_df.groupby(
    ["client_hash_id", "content_hash_id", "report_date"]
).size()

print(grain_check.head())

print("Duplicate grain records:", (grain_check > 1).sum())

client_hash_id           content_hash_id           report_date
client_0797ff3a1fc9a6a5  content_004e9c4c32e88631  2026-03-02     1
                                                   2026-03-04     1
                                                   2026-03-05     1
                                                   2026-03-06     1
                                                   2026-03-07     1
dtype: int64
Duplicate grain records: 0


In [16]:
# Verify March 2026 time window

print("Minimum date:", performance_df["report_date"].min())
print("Maximum date:", performance_df["report_date"].max())

Minimum date: 2026-03-01
Maximum date: 2026-03-31


##2. Fields

### Features

The following fields are used as historical signals:

- gsc_clicks
- gsc_impressions
- gsc_avg_position
- ga4_sessions
- ga4_engagement_rate

These represent observed search visibility, traffic, and engagement behaviour.

### Label / Proxy

Refresh Opportunity Score

This is a ranking signal created from historical performance to identify content that may benefit from optimization.

### Context Fields

- client_hash_id
- content_hash_id
- report_date

These fields identify records and organize the panel but are not model features.

### Excluded

The following are excluded:

- Future performance information
- Rows where ga4_data_available = FALSE
- Label-derived columns

These exclusions reduce data leakage risk.

In [17]:
# Verify important fields exist

required_fields = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

missing_fields = [
    col for col in required_fields
    if col not in performance_df.columns
]

print("Missing fields:", missing_fields)

Missing fields: []


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
# Query 1: Count rows in March 2026

march_rows = len(performance_df)

print("March 2026 rows:", march_rows)

March 2026 rows: 9200000


In [19]:
# Query 2: Validate grain

grain = performance_df.groupby(
    ["client_hash_id", "content_hash_id", "report_date"]
).size()

duplicate_rows = (grain > 1).sum()

print("Duplicate grain combinations:", duplicate_rows)

Duplicate grain combinations: 0


In [20]:
# Query 3: Check GA4 availability and missing values

print("GA4 availability:")
print(performance_df["ga4_data_available"].value_counts())

print("\nMissing values:")
print(performance_df.isnull().sum().sort_values(ascending=False).head(10))

GA4 availability:
ga4_data_available
False    6071395
True      398904
Name: count, dtype: int64

Missing values:
gsc_avg_position      5823382
ga4_data_available    2729701
ga4_sessions          2729701
ga4_pageviews         2729701
ga4_users             2729701
ai_chatgpt            2729701
sessions_ai           2729701
sessions_paid         2729701
sessions_social       2729701
sessions_referral     2729701
dtype: int64


## 4. Data limits

This dataset has several limitations:

- Historical performance does not guarantee future results.
- Some rows contain GSC-only data where GA4 metrics are unavailable.
- Daily observations create overlapping time windows.
- The data supports ranking and decision support but does not prove causation.

In [21]:
# Check available dates and data coverage

print(performance_df["report_date"].describe())

count        9200000
unique            31
top       2026-03-30
freq          331230
Name: report_date, dtype: object


##5. Feature Engineering

The following features are created from historical performance signals. These features are decision-support signals for identifying content refresh opportunities.

Features:

1. CTR (Click Through Rate)
- Measures clicks generated from impressions.

2. Engagement Rate
- Measures engaged sessions compared with total sessions.

3. AI Traffic Share
- Measures the proportion of sessions coming from AI sources.

4. Content Visibility Score
- Combines impressions and clicks to represent search visibility.

5. Position Score
- Converts average search position into a higher-is-better signal.

In [22]:
# Create five features

# 1. Click Through Rate
performance_df["ctr"] = (
    performance_df["gsc_clicks"] /
    performance_df["gsc_impressions"].replace(0, 1)
)

# 2. Engagement Rate
performance_df["engagement_rate"] = (
    performance_df["ga4_engaged_sessions"] /
    performance_df["ga4_sessions"].replace(0, 1)
)

# 3. AI Traffic Share
performance_df["ai_traffic_share"] = (
    performance_df["sessions_ai"] /
    (
        performance_df["ga4_sessions"].replace(0, 1)
    )
)

# 4. Content Visibility Score
performance_df["visibility_score"] = (
    performance_df["gsc_impressions"] +
    performance_df["gsc_clicks"]
)

# 5. Position Score
performance_df["position_score"] = (
    1 /
    performance_df["gsc_avg_position"].replace(0, 1)
)

print("Features created successfully")

performance_df[
    [
        "ctr",
        "engagement_rate",
        "ai_traffic_share",
        "visibility_score",
        "position_score"
    ]
].head()

Features created successfully


,ctr,engagement_rate,ai_traffic_share,visibility_score,position_score
0,0.0,0.0,0.0,0,NaN
1,0.0,0.0,0.0,0,NaN
2,0.0,0.0,0.0,2,0.105263
3,0.0,0.0,0.0,0,NaN
4,0.0,0.0,0.0,2,0.166667


In [23]:
performance_df.shape

(9200000, 35)

##6. Refresh Opportunity Score (Proxy Label)

The refresh opportunity score is a decision-support ranking signal created from historical performance.

It combines:
- Low search visibility
- Low clicks compared with impressions
- Lower engagement

A higher score indicates content that may have more opportunity for improvement.

This is used as a proxy label because the dataset does not contain a direct human-labelled refresh outcome.

In [24]:
# Create Refresh Opportunity Score

performance_df["refresh_opportunity_score"] = (
    (1 - performance_df["ctr"].clip(0, 1))
    +
    (1 - performance_df["engagement_rate"].clip(0, 1))
    +
    (1 / performance_df["gsc_avg_position"].replace(0, 1))
)

performance_df[
    [
        "content_hash_id",
        "ctr",
        "engagement_rate",
        "gsc_avg_position",
        "refresh_opportunity_score"
    ]
].head()

,content_hash_id,ctr,engagement_rate,gsc_avg_position,refresh_opportunity_score
0,content_bf284ffdacec439c,0.0,0.0,NaN,NaN
1,content_83c3b55f6607b402,0.0,0.0,NaN,NaN
2,content_e9e39c41d7e62b7a,0.0,0.0,9.5,2.105263
3,content_0de9f0ee1ef10d09,0.0,0.0,NaN,NaN
4,content_7469bc98ef7f1737,0.0,0.0,6.0,2.166667


In [25]:
performance_df["refresh_opportunity_score"].describe()

,refresh_opportunity_score
count,1.987587e+06
mean,2.265893e+00
std,9.968351e-01
min,1.851852e-02
25%,2.043808e+00
50%,2.116646e+00
75%,2.238095e+00
max,4.460000e+02


In [26]:
top_refresh = performance_df.sort_values(
    "refresh_opportunity_score",
    ascending=False
).head(10)

top_refresh[
    [
        "content_hash_id",
        "refresh_opportunity_score"
    ]
]

,content_hash_id,refresh_opportunity_score
7401921,content_1ebd9dbac77bbbeb,446.000000
6632841,content_fd1d19e381fc653e,325.769231
3855031,content_96f4295b7b3218b3,206.666667
9063723,content_7ea6d35866735527,192.000000
2576616,content_7c0175b190d7c6e7,190.250000
1337448,content_05482eae6a287d0c,141.750000
1005539,content_738f3a26084f61b2,133.600000
7729396,content_4a9231861a28d3ff,131.500000
4105588,content_7d71db9cad5304cf,127.000000
3899548,content_ecc27d32010b18e0,125.000000


## Self-check


Before submitting, confirm each line honestly:

✅ Every section above is filled — markdown thinking and the code that backs it.

✅ The notebook runs top to bottom with no errors (Runtime → Run all).

✅ No client names, URLs, or private queries are included anywhere.

✅ My claims use careful words: observed, measured, directional, decision-support.

✅ The notebook is committed to my repo under work/notebooks/ and the repository URL is submitted on the card.